# 제7장: 사례 연구

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch07.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### Case 연구: 한국 기업의 AI 거버넌스 실천

## 금융 사례: KBank의 AI 신용평가 거버넌스

### 도입 배경

### 거버넌스 체계 설계

### 공정성 관리 체계

**K-Bank Style Fairness Evaluation Framework**

In [ ]:
"""
Financial AI Credit Scoring Fairness Evaluation Framework
Implemented based on K-Bank case (Generalised)
- Prohibited variable check
- Proxy discrimination analysis
- 4/5 rule evaluation
- Adverse Action explanation generation
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd

@dataclass
class CreditFairnessConfig:
 """Credit Scoring Fairness Configuration"""
 
 # Prohibited variables (Based on Credit Information Act)
 prohibited_variables: List[str] = field(default_factory=lambda: [
 'gender', 'sex',
 'birth_region', 'origin_region',
 'marital_status',
 'religion',
 'political_view',
 ])
 
 # Proxy suspect variables
 proxy_suspects: List[str] = field(default_factory=lambda: [
 'zip_code', 'postal_code',
 'university', 'college',
 'employer_industry',
 ])
 
 # Protected attributes for analysis
 protected_attributes: List[str] = field(default_factory=lambda: [
 'age_group', # Young/Middle/Senior
 'region', # Urban/Rural
 'income_level', # Low/Mid/High Income
 ])
 
 # Thresholds
 dp_ratio_threshold: float = 0.8
 proxy_correlation_threshold: float = 0.3


class CreditScoringFairnessEvaluator:
 """
 Credit Scoring AI Fairness Evaluator
 Compliant with FSC guidelines and Credit Information Act
 """
 
 def __init__(self, config: Optional[CreditFairnessConfig] = None):
 self.config = config or CreditFairnessConfig()
 
 def validate_features(
 self,
 feature_names: List[str],
 ) -> Dict[str, List[str]]:
 """Feature validation: Check prohibited and proxy suspect variables"""
 
 prohibited_found = []
 proxy_found = []
 
 for feat in feature_names:
 feat_lower = feat.lower()
 
 # Check prohibited variables
 for prohibited in self.config.prohibited_variables:
 if prohibited in feat_lower:
 prohibited_found.append(feat)
 break
 
 # Check proxy suspect variables
 for proxy in self.config.proxy_suspects:
 if proxy in feat_lower:
 proxy_found.append(feat)
 break
 
 return {
 'prohibited': prohibited_found,
 'proxy_suspects': proxy_found,
 'clean': [f for f in feature_names 
 if f not in prohibited_found and f not in proxy_found],
 }
 
 def analyze_proxy_discrimination(
 self,
 df: pd.DataFrame,
 feature_name: str,
 protected_attribute: str,
 ) -> Dict:
 """Proxy discrimination analysis"""
 
 # Correlation analysis
 if df[feature_name].dtype == 'object':
 # Categorical: Cramer's V
 from scipy.stats import chi2_contingency
 contingency = pd.crosstab(df[feature_name], df[protected_attribute])
 chi2, p_value, dof, expected = chi2_contingency(contingency)
 n = len(df)
 min_dim = min(contingency.shape) - 1
 cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
 correlation = cramers_v
 else:
 # Numerical: Point Biserial or Spearman
 if df[protected_attribute].nunique() == 2:
 from scipy.stats import pointbiserialr
 # Binary conversion
 binary = pd.factorize(df[protected_attribute])[0]
 corr, p_value = pointbiserialr(binary, df[feature_name])
 correlation = abs(corr)
 else:
 correlation = abs(df[feature_name].corr(
 pd.factorize(df[protected_attribute])[0]
 ))
 
 is_proxy = correlation >= self.config.proxy_correlation_threshold
 
 return {
 'feature': feature_name,
 'protected_attribute': protected_attribute,
 'correlation': correlation,
 'is_proxy': is_proxy,
 'recommendation': 'Remove or Use with Caution' if is_proxy else 'Safe to Use',
 }
 
 def evaluate_approval_rates(
 self,
 y_pred: np.ndarray,
 sensitive_features: pd.DataFrame,
 ) -> Dict[str, Dict]:
 """Approval rate analysis by group (4/5 rule)"""
 
 results = {}
 
 for attr in self.config.protected_attributes:
 if attr not in sensitive_features.columns:
 continue
 
 groups = sensitive_features[attr].unique()
 group_rates = {}
 
 for group in groups:
 mask = sensitive_features[attr] == group
 approval_rate = y_pred[mask].mean()
 group_rates[group] = {
 'approval_rate': approval_rate,
 'count': mask.sum(),
 'approved': y_pred[mask].sum(),
 }
 
 # 4/5 rule evaluation
 rates = [g['approval_rate'] for g in group_rates.values()]
 max_rate = max(rates) if rates else 0
 
 for group, stats in group_rates.items():
 stats['ratio_to_max'] = stats['approval_rate'] / max_rate if max_rate > 0 else 1.0
 stats['passes_4_5'] = stats['ratio_to_max'] >= 0.8
 
 results[attr] = {
 'groups': group_rates,
 'max_rate': max_rate,
 'overall_pass': all(g['passes_4_5'] for g in group_rates.values()),
 'min_ratio': min(g['ratio_to_max'] for g in group_rates.values()),
 }
 
 return results
 
 def generate_adverse_action_notice(
 self,
 shap_values: Dict[str, float],
 feature_descriptions: Dict[str, str],
 customer_name: str = "Customer",
 application_id: str = "",
 ) -> str:
 """
 Generate Adverse Action Notice
 Compliance with Article 36-2 of the Credit Information Act and ECOA Regulation B
 """
 
 # Negative contributing factors (SHAP < 0)
 negative_factors = sorted(
 [(k, v) for k, v in shap_values.items() if v < 0],
 key=lambda x: x[1]
 )[:4] # Top 4
 
 notice = f"""

 Credit Decision Notice


Dear {customer_name},

Application No: {application_id}
Decision Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}

After careful review of your credit application,
we regret to inform you that we are unable to approve it at this time.



 Key Influencing Factors

The following factors primarily influenced this decision:

"""
 
 for i, (feat, _) in enumerate(negative_factors, 1):
 desc = feature_descriptions.get(feat, feat)
 notice += f" {i}. {desc}\n"
 
 notice += """


 Your Rights

1. Request for Further Explanation
 - You may request a detailed explanation of this decision.

2. Right to Appeal
 - If you disagree with this decision, you may appeal within 30 days.

3. Request for Human Review
 - You may request a review by a credit officer.

4. Credit Information Inquiry
 - You may check your credit info once a year for free.



 Contact Us

Customer Center: 1234-5678 (Weekdays 09:00-18:00)
Email: credit@kbank.co.kr
Website: www.kbank.co.kr/credit-decision



This notice is sent in accordance with Article 36-2 of the 
Act on the Use and Protection of Credit Information.

K-Bank Credit Approval Department
"""
 
 return notice
 
 def generate_regulatory_report(
 self,
 model_name: str,
 evaluation_period: str,
 approval_results: Dict,
 total_applications: int,
 ) -> str:
 """Generate report for Financial Supervisory Service submission"""
 
 report = f"""
# AI Credit Scoring Model Fairness Evaluation Report

Model Name: {model_name} 
Evaluation Period: {evaluation_period} 
Total Reviews: {total_applications:,} 
Report Date: {pd.Timestamp.now().strftime('%Y-%m-%d')}

---

## 1. Overview

This report presents the fairness evaluation results for the AI credit scoring model
in compliance with the FSC "AI Guidelines for the Financial Sector" and
the "Act on the Use and Protection of Credit Information".

---

## 2. Approval Rate Analysis by Group

"""
 
 overall_pass = True
 
 for attr, result in approval_results.items():
 attr_pass = result.get('overall_pass', False)
 overall_pass = overall_pass and attr_pass
 
 report += f"### {attr}\n\n"
 report += f"4/5 Rule Met: {'[PASS] Yes' if attr_pass else '[FAIL] No'}\n"
 report += f"Minimum Ratio: {result.get('min_ratio', 0):.4f}\n\n"
 
 report += "| Group | Reviews | Approved | Approval Rate | Ratio to Max | 4/5 Rule |\n"
 report += "|------|----------|----------|--------|--------------|----------|\n"
 
 for group, stats in result.get('groups', {}).items():
 passed = '[PASS]' if stats['passes_4_5'] else '[FAIL]'
 report += f"| {group} | {stats['count']:,} | {stats['approved']:,} | {stats['approval_rate']:.2%} | {stats['ratio_to_max']:.4f} | {passed} |\n"
 
 report += "\n"
 
 report += f"""
---

## 3. Comprehensive Evaluation

| Evaluation Item | Result |
|----------|------|
| Use of Prohibited Variables | [PASS] None |
| Proxy Discrimination Risk | [WARN] Under Monitoring |
| 4/5 Rule Met Overall | {'[PASS] Met' if overall_pass else '[FAIL] Not Met'} |
| Explanation System | [PASS] Operational |

---

## 4. Required Actions

"""
 
 if not overall_pass:
 report += """
### Immediate Action Required

The following actions are applied for groups not meeting the 4/5 rule:

1. Increase human review rate for affected groups (30% -> 50%)
2. Review data augmentation for affected group data during retraining
3. Strengthen monthly fairness monitoring

"""
 else:
 report += """
### Status Quo Maintained

All fairness criteria are currently met.
Maintain regular quarterly monitoring.

"""
 
 report += """
---

## 5. Attachments

1. Detailed Statistical Data (Attachment 1)
2. Model Card (Attachment 2)
3. Fairness Evaluation Methodology (Attachment 3)

---

Author: AI Ethics Team 
Reviewer: Head of Model Risk Management 
Approver: CRO

*Retained for 5 years per Article 15.*
"""
 
 return report

### 성능 및 교훈

## 의료 사례: AUniversityHospital의 의료 AI 거버넌스

### 도입 배경

### 거버넌스 체계

### 임상 검증 및 모니터링

**Healthcare AI Clinical Monitoring System**

In [ ]:
"""
Healthcare AI Clinical Monitoring System

Implementation based on A-University Hospital case (Generalized)
- Interpretation Discrepancy Analysis
- Sub-group Performance Monitoring
- Adverse Case Tracking
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from enum import Enum

class DiscrepancyType(Enum):
 """ days """
 AI_POSITIVE_PHYSICIAN_NEGATIVE = "AI , "
 AI_NEGATIVE_PHYSICIAN_POSITIVE = "AI , "
 SEVERITY_DISAGREEMENT = " Decision days "
 LOCATION_DISAGREEMENT = " Location days "

@dataclass
class ClinicalCase:
 """Clinical """
 case_id: str
 patient_id: str
 modality: str # X-ray, CT, MRI 
 ai_result: Dict # AI physician_result: Dict # final_diagnosis: Optional[Dict] = None # Final (Tracking )
 timestamp: datetime = field(default_factory=datetime.now)
 
 # information ( )
 patient_age: int = 0
 patient_sex: str = ""
 comorbidities: List[str] = field(default_factory=list)


class MedicalAIClinicalMonitor:
 """
 Healthcare AI Clinical Monitoring
 
 Performance Tracking, days Analysis, Monitoring
 """
 
 def __init__(
 self,
 system_name: str,
 sensitivity_threshold: float = 0.90,
 specificity_threshold: float = 0.85,
 ):
 self.system_name = system_name
 self.sensitivity_threshold = sensitivity_threshold
 self.specificity_threshold = specificity_threshold
 
 self.cases: List[ClinicalCase] = []
 self.alerts: List[Dict] = []
 
 def record_case(self, case: ClinicalCase) -> None:
 """ """
 self.cases.append(case)
 
 # days discrepancy = self._check_discrepancy(case)
 if discrepancy:
 self._handle_discrepancy(case, discrepancy)
 
 def _check_discrepancy(self, case: ClinicalCase) -> Optional[DiscrepancyType]:
 """ days """
 
 ai_positive = case.ai_result.get('finding_positive', False)
 physician_positive = case.physician_result.get('finding_positive', False)
 
 if ai_positive and not physician_positive:
 return DiscrepancyType.AI_POSITIVE_PHYSICIAN_NEGATIVE
 elif not ai_positive and physician_positive:
 return DiscrepancyType.AI_NEGATIVE_PHYSICIAN_POSITIVE
 
 return None
 
 def _handle_discrepancy(
 self,
 case: ClinicalCase,
 discrepancy: DiscrepancyType,
 ) -> None:
 """ days """
 
 alert = {
 'case_id': case.case_id,
 'timestamp': datetime.now(),
 'discrepancy_type': discrepancy.value,
 'severity': 'high' if discrepancy == DiscrepancyType.AI_NEGATIVE_PHYSICIAN_POSITIVE else 'medium',
 'ai_result': case.ai_result,
 'physician_result': case.physician_result,
 }
 
 self.alerts.append(alert)
 
 # AI , ( )
 if discrepancy == DiscrepancyType.AI_NEGATIVE_PHYSICIAN_POSITIVE:
 self._escalate_alert(alert)
 
 def _escalate_alert(self, alert: Dict) -> None:
 """ """
 # Implementation: days, SMS, print(f"[CRITICAL] : {alert['case_id']}")
 
 def calculate_performance(
 self,
 start_date: Optional[datetime] = None,
 end_date: Optional[datetime] = None,
 ) -> Dict:
 """Period Performance Calculation"""
 
 # Final evaluated_cases = [
 c for c in self.cases
 if c.final_diagnosis is not None
 ]
 
 if start_date:
 evaluated_cases = [c for c in evaluated_cases if c.timestamp >= start_date]
 if end_date:
 evaluated_cases = [c for c in evaluated_cases if c.timestamp <= end_date]
 
 if not evaluated_cases:
 return {'error': 'No evaluated cases'}
 
 # AI vs Final 
 ai_positive = np.array([c.ai_result.get('finding_positive', False) for c in evaluated_cases])
 final_positive = np.array([c.final_diagnosis.get('confirmed_positive', False) for c in evaluated_cases])
 
 tp = ((ai_positive == True) & (final_positive == True)).sum()
 fn = ((ai_positive == False) & (final_positive == True)).sum()
 fp = ((ai_positive == True) & (final_positive == False)).sum()
 tn = ((ai_positive == False) & (final_positive == False)).sum()
 
 sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
 specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
 ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
 npv = tn / (tn + fn) if (tn + fn) > 0 else 0
 
 return {
 'n_cases': len(evaluated_cases),
 'sensitivity': sensitivity,
 'specificity': specificity,
 'ppv': ppv,
 'npv': npv,
 'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn,
 'meets_sensitivity_threshold': sensitivity >= self.sensitivity_threshold,
 'meets_specificity_threshold': specificity >= self.specificity_threshold,
 }
 
 def analyze_subgroups(
 self,
 subgroup_column: str,
 ) -> Dict[str, Dict]:
 """ Performance Analysis"""
 
 evaluated_cases = [c for c in self.cases if c.final_diagnosis is not None]
 
 if subgroup_column == 'age_group':
 # Classification
 def get_age_group(age):
 if age < 40:
 return '40 '
 elif age < 60:
 return '40-59 '
 elif age < 80:
 return '60-79 '
 else:
 return '80 '
 
 groups = {}
 for case in evaluated_cases:
 group = get_age_group(case.patient_age)
 if group not in groups:
 groups[group] = []
 groups[group].append(case)
 
 elif subgroup_column == 'sex':
 groups = {' ': [], ' ': []}
 for case in evaluated_cases:
 if case.patient_sex in ['M', 'male', ' ']:
 groups[' '].append(case)
 elif case.patient_sex in ['F', 'female', ' ']:
 groups[' '].append(case)
 
 else:
 return {'error': f'Unknown subgroup column: {subgroup_column}'}
 
 # Performance Calculation
 results = {}
 for group_name, cases in groups.items():
 if len(cases) < 30: # Minimum 
 results[group_name] = {'n': len(cases), 'insufficient_data': True}
 continue
 
 ai_pos = np.array([c.ai_result.get('finding_positive', False) for c in cases])
 final_pos = np.array([c.final_diagnosis.get('confirmed_positive', False) for c in cases])
 
 tp = ((ai_pos == True) & (final_pos == True)).sum()
 fn = ((ai_pos == False) & (final_pos == True)).sum()
 fp = ((ai_pos == True) & (final_pos == False)).sum()
 tn = ((ai_pos == False) & (final_pos == False)).sum()
 
 sens = tp / (tp + fn) if (tp + fn) > 0 else 0
 spec = tn / (tn + fp) if (tn + fp) > 0 else 0
 
 results[group_name] = {
 'n': len(cases),
 'sensitivity': sens,
 'specificity': spec,
 'positive_rate': final_pos.mean(),
 }
 
 # Analysis
 sensitivities = [r['sensitivity'] for r in results.values() 
 if not r.get('insufficient_data', False)]
 if len(sensitivities) >= 2:
 max_gap = max(sensitivities) - min(sensitivities)
 results['_gap_analysis'] = {
 'max_sensitivity_gap': max_gap,
 'equity_concern': max_gap > 0.10,
 }
 
 return results
 
 def generate_monthly_report(self, month: str) -> str:
 """ Clinical Performance Report Generation"""
 
 perf = self.calculate_performance()
 
 report = f"""
# {self.system_name} Clinical Performance Report

Period: {month} 
 : {perf.get('n_cases', 0):,} 

---

## 1. Performance Metric

| Metric | | | |
|------|-----|------|------|
| (Sensitivity) | {perf.get('sensitivity', 0):.2%} | >={self.sensitivity_threshold:.0%} | {'[PASS]' if perf.get('meets_sensitivity_threshold', False) else '[FAIL]'} |
| (Specificity) | {perf.get('specificity', 0):.2%} | >={self.specificity_threshold:.0%} | {'[PASS]' if perf.get('meets_specificity_threshold', False) else '[FAIL]'} |
| (PPV) | {perf.get('ppv', 0):.2%} | - | - |
| (NPV) | {perf.get('npv', 0):.2%} | - | - |

---

## 2. | | AI | AI |
|--|---------|---------|
| Final | {perf.get('tp', 0)} | {perf.get('fn', 0)} |
| Final | {perf.get('fp', 0)} | {perf.get('tn', 0)} |

---

## 3. Interpretation Discrepancy Analysis

| days | |
|------------|------|
"""
 
 # days discrepancy_counts = {}
 for alert in self.alerts:
 dtype = alert['discrepancy_type']
 discrepancy_counts[dtype] = discrepancy_counts.get(dtype, 0) + 1
 
 for dtype, count in discrepancy_counts.items():
 report += f"| {dtype} | {count} |\n"
 
 report += f"""

---

## 4. Performance

### 
"""
 
 age_analysis = self.analyze_subgroups('age_group')
 report += "| | N | | |\n"
 report += "|--------|---|--------|--------|\n"
 for group, stats in age_analysis.items():
 if group.startswith('_'):
 continue
 if stats.get('insufficient_data', False):
 report += f"| {group} | {stats['n']} | Data | - |\n"
 else:
 report += f"| {group} | {stats['n']} | {stats['sensitivity']:.2%} | {stats['specificity']:.2%} |\n"
 
 report += """

---

## 5. """
 
 if perf.get('fn', 0) > 0:
 report += f"- [WARN] {perf['fn']} Analysis \n"
 
 if not perf.get('meets_sensitivity_threshold', True):
 report += "- [FAIL] : Model \n"
 
 gap_analysis = age_analysis.get('_gap_analysis', {})
 if gap_analysis.get('equity_concern', False):
 report += f"- [WARN] {gap_analysis['max_sensitivity_gap']:.1%}: Equity \n"
 
 report += """

---

 : AI Clinical 
 : 
Approval: 

* Report Healthcare 15 5years .*
"""
 
 return report

### 성능 및 교훈

## Public 사례: H공공기관의 AI 행정 서비스

### 도입 배경

### 거버넌스 체계

**Public Service AI Governance System**

In [ ]:
"""
Public Service AI Governance System

Implementation based on H-Public Institution case (Generalized)
- Discretionary Action Judgment
- Guarantee of Citizen Rights
- Equity Monitoring
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from enum import Enum
import pandas as pd

class DecisionType(Enum):
 """ """
 BOUND = " " # DISCRETIONARY = "Discretionary " # Discretionary 

class OversightLevel(Enum):
 """ Level"""
 FULL_AUTO = " " # HUMAN_APPROVAL = " Approval" # Approval AI_SUPPORT = "AI Support" # AI information @dataclass
class CitizenRequest:
 """ """
 request_id: str
 citizen_id: str # 
 request_type: str
 submitted_data: Dict
 timestamp: datetime = field(default_factory=datetime.now)
 
 # information
 ai_recommendation: Optional[Dict] = None
 human_decision: Optional[Dict] = None
 final_decision: Optional[Dict] = None
 
 # information (Equity Analysis , )
 region: str = ""
 age_group: str = ""
 income_level: str = ""

@dataclass
class AppealRequest:
 """ """
 appeal_id: str
 original_request_id: str
 citizen_id: str
 appeal_reason: str
 submitted_at: datetime = field(default_factory=datetime.now)
 
 # review_result: Optional[str] = None
 reviewed_by: Optional[str] = None
 reviewed_at: Optional[datetime] = None


class PublicServiceAIGovernance:
 """
 Public AI Governance
 
 20 , AI Compliance
 """
 
 def __init__(self, service_name: str):
 self.service_name = service_name
 self.requests: List[CitizenRequest] = []
 self.appeals: List[AppealRequest] = []
 
 def classify_decision_type(
 self,
 request_type: str,
 has_discretion: bool,
 ) -> Tuple[DecisionType, OversightLevel]:
 """
 Level Classification
 
 20 Discretionary Automation 
 """
 
 if has_discretion:
 return DecisionType.DISCRETIONARY, OversightLevel.AI_SUPPORT
 else:
 return DecisionType.BOUND, OversightLevel.HUMAN_APPROVAL
 
 def process_request(
 self,
 request: CitizenRequest,
 ai_model,
 decision_type: DecisionType,
 oversight: OversightLevel,
 ) -> Dict:
 """ """
 
 # AI Generation
 ai_recommendation = ai_model.predict(request.submitted_data)
 request.ai_recommendation = {
 'recommendation': ai_recommendation,
 'confidence': ai_model.get_confidence(),
 'explanation': ai_model.get_explanation(),
 'generated_at': datetime.now().isoformat(),
 }
 
 # Level if oversight == OversightLevel.FULL_AUTO:
 # Automation ( )
 if decision_type != DecisionType.BOUND:
 raise ValueError("Discretionary Automation ( 20 )")
 
 request.final_decision = {
 'decision': ai_recommendation,
 'decision_type': 'automated',
 'legal_basis': ' 20 1 ',
 }
 
 else:
 # request.final_decision = {
 'decision': None, # 'decision_type': 'pending_human_review',
 'ai_recommendation': ai_recommendation,
 }
 
 self.requests.append(request)
 
 return {
 'request_id': request.request_id,
 'ai_recommendation': ai_recommendation,
 'oversight_level': oversight.value,
 'requires_human_review': oversight != OversightLevel.FULL_AUTO,
 }
 
 def record_human_decision(
 self,
 request_id: str,
 decision: str,
 decided_by: str,
 override_ai: bool = False,
 override_reason: str = "",
 ) -> None:
 """ """
 
 request = next((r for r in self.requests if r.request_id == request_id), None)
 if not request:
 raise ValueError(f"Request not found: {request_id}")
 
 request.human_decision = {
 'decision': decision,
 'decided_by': decided_by,
 'decided_at': datetime.now().isoformat(),
 'override_ai': override_ai,
 'override_reason': override_reason if override_ai else None,
 }
 
 request.final_decision = {
 'decision': decision,
 'decision_type': 'human_reviewed',
 'ai_recommendation': request.ai_recommendation['recommendation'],
 'ai_followed': not override_ai,
 }
 
 def process_appeal(
 self,
 appeal: AppealRequest,
 ) -> Dict:
 """ """
 
 self.appeals.append(appeal)
 
 # return {
 'appeal_id': appeal.appeal_id,
 'status': 'assigned_for_review',
 'expected_response_days': 14, # 14days 'citizen_rights': [
 ' ',
 'Addition ',
 ' ',
 ],
 }
 
 def analyze_equity(self) -> Dict:
 """Equity Analysis"""
 
 if not self.requests:
 return {'error': 'No data'}
 
 # Analysis
 region_stats = {}
 for request in self.requests:
 if not request.final_decision:
 continue
 
 region = request.region or 'unknown'
 if region not in region_stats:
 region_stats[region] = {'total': 0, 'approved': 0}
 
 region_stats[region]['total'] += 1
 if request.final_decision.get('decision') == 'approved':
 region_stats[region]['approved'] += 1
 
 # Approval Calculation
 for region, stats in region_stats.items():
 stats['approval_rate'] = stats['approved'] / stats['total'] if stats['total'] > 0 else 0
 
 # Analysis
 rates = [s['approval_rate'] for s in region_stats.values() if s['total'] >= 30]
 if len(rates) >= 2:
 max_gap = max(rates) - min(rates)
 equity_concern = max_gap > 0.15 # 15%p else:
 max_gap = 0
 equity_concern = False
 
 return {
 'by_region': region_stats,
 'max_gap': max_gap,
 'equity_concern': equity_concern,
 }
 
 def generate_transparency_report(self) -> str:
 """ Report"""
 
 total_requests = len(self.requests)
 decided_requests = [r for r in self.requests if r.final_decision]
 
 # AI ai_followed = sum(
 1 for r in decided_requests 
 if r.final_decision.get('ai_followed', False)
 )
 ai_follow_rate = ai_followed / len(decided_requests) if decided_requests else 0
 
 # total_appeals = len(self.appeals)
 resolved_appeals = [a for a in self.appeals if a.review_result]
 upheld_appeals = sum(1 for a in resolved_appeals if a.review_result == 'upheld')
 upheld_rate = upheld_appeals / len(resolved_appeals) if resolved_appeals else 0
 
 equity = self.analyze_equity()
 
 report = f"""
# {self.service_name} Report

Period: {datetime.now().strftime('%Yyears %m ')} 
days: {datetime.now().strftime('%Yyears %m %ddays')}

---

## 1. | Items | |
|------|------|
| | {total_requests:,} |
| | {len(decided_requests):,} |
| AI | {ai_follow_rate:.1%} |

---

## 2. | Items | / |
|------|----------|
| | {total_appeals:,} |
| | {len(resolved_appeals):,} |
| | {upheld_rate:.1%} |

---

## 3. Equity Analysis

### Approval 

| | | Approval |
|------|----------|--------|
"""
 
 for region, stats in equity.get('by_region', {}).items():
 if stats['total'] >= 30:
 report += f"| {region} | {stats['total']:,} | {stats['approval_rate']:.1%} |\n"
 
 if equity.get('equity_concern', False):
 report += f"\n[WARN] Approval ({equity['max_gap']:.1%}p) (15%p) . Analysis .\n"
 
 report += f"""

---

## 4. :

1.  : AI based Definition Major .
2.  : 14days .
3.  : .
4. information: Addition information .

---

 : {self.service_name} (1234-5678)

* Report PublicInstitution information .*
"""
 
 return report

### 성능 및 교훈

## 제조 사례: MHeavy Industries의 AI 품질 관리

### 도입 배경

### 거버넌스 체계

**Manufacturing AI Quality Management System**

In [ ]:
"""
Manufacturing AI Quality Management System

Implementation based on M-Heavy Industries case (Generalized)
- Welding Defect Inspection
- Quality Record Management
- Skilled Worker Verification System
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from enum import Enum
import numpy as np

class DefectSeverity(Enum):
 """ ( )"""
 NONE = " None"
 MINOR = " ( )"
 MAJOR = " ( )"
 CRITICAL = " ( )"

class InspectionResult(Enum):
 """ """
 PASS = " "
 CONDITIONAL_PASS = " "
 FAIL = " "
 REINSPECTION = " "

@dataclass
class WeldInspection:
 """ """
 inspection_id: str
 work_order: str # weld_location: str # Location Code
 welder_id: str # ID
 
 # AI ai_result: Dict = field(default_factory=dict)
 ai_confidence: float = 0.0
 ai_defects_detected: List[Dict] = field(default_factory=list)
 
 # Skilled Worker Verification
 inspector_id: Optional[str] = None
 inspector_result: Optional[Dict] = None
 inspector_override: bool = False
 override_reason: str = ""
 
 # Final 
 final_result: Optional[InspectionResult] = None
 
 timestamp: datetime = field(default_factory=datetime.now)


class ManufacturingQualityAI:
 """
 Manufacturing AI Management
 
 Welding Defect Inspection, Quality Record Management
 """
 
 def __init__(
 self,
 system_name: str,
 min_detection_rate: float = 0.99, # Minimum 
 max_overkill_rate: float = 0.05, # ):
 self.system_name = system_name
 self.min_detection_rate = min_detection_rate
 self.max_overkill_rate = max_overkill_rate
 
 self.inspections: List[WeldInspection] = []
 
 def classify_defect_severity(
 self,
 defect_type: str,
 defect_size_mm: float,
 weld_class: str, # A, B, C ( )
 ) -> DefectSeverity:
 """ Classification ( )"""
 
 # (Example)
 thresholds = {
 'A': {'minor': 2.0, 'major': 4.0}, # Class A: 'B': {'minor': 3.0, 'major': 6.0},
 'C': {'minor': 5.0, 'major': 10.0},
 }
 
 thresh = thresholds.get(weld_class, thresholds['B'])
 
 if defect_size_mm <= thresh['minor']:
 return DefectSeverity.MINOR
 elif defect_size_mm <= thresh['major']:
 return DefectSeverity.MAJOR
 else:
 return DefectSeverity.CRITICAL
 
 def process_inspection(
 self,
 inspection: WeldInspection,
 ) -> Dict:
 """ """
 
 # AI based 
 if inspection.ai_confidence >= 0.95:
 # : AI based Classification
 if not inspection.ai_defects_detected:
 routing = "auto_pass" # else:
 max_severity = max(
 (self.classify_defect_severity(
 d['type'], d['size_mm'], d.get('weld_class', 'B')
 ) for d in inspection.ai_defects_detected),
 default=DefectSeverity.NONE
 )
 if max_severity == DefectSeverity.CRITICAL:
 routing = "immediate_review" # else:
 routing = "standard_review" # days else:
 # : routing = "low_confidence_review"
 
 self.inspections.append(inspection)
 
 return {
 'inspection_id': inspection.inspection_id,
 'ai_confidence': inspection.ai_confidence,
 'defects_count': len(inspection.ai_defects_detected),
 'routing': routing,
 'requires_human_review': routing != "auto_pass",
 }
 
 def record_inspector_decision(
 self,
 inspection_id: str,
 inspector_id: str,
 result: InspectionResult,
 override_ai: bool = False,
 override_reason: str = "",
 additional_defects: List[Dict] = None,
 ) -> None:
 """Skilled Worker Verification """
 
 inspection = next(
 (i for i in self.inspections if i.inspection_id == inspection_id),
 None
 )
 if not inspection:
 raise ValueError(f"Inspection not found: {inspection_id}")
 
 inspection.inspector_id = inspector_id
 inspection.inspector_result = {
 'result': result.value,
 'override_ai': override_ai,
 'override_reason': override_reason,
 'additional_defects': additional_defects or [],
 'timestamp': datetime.now().isoformat(),
 }
 inspection.inspector_override = override_ai
 inspection.override_reason = override_reason
 inspection.final_result = result
 
 def calculate_performance(self) -> Dict:
 """AI Performance Calculation"""
 
 # Skilled Worker Verification verified = [i for i in self.inspections if i.inspector_result]
 
 if not verified:
 return {'error': 'No verified inspections'}
 
 # AI vs Comparison
 ai_detected_defect = np.array([
 len(i.ai_defects_detected) > 0 for i in verified
 ])
 inspector_confirmed_defect = np.array([
 i.final_result in [InspectionResult.FAIL, InspectionResult.CONDITIONAL_PASS]
 for i in verified
 ])
 
 tp = ((ai_detected_defect == True) & (inspector_confirmed_defect == True)).sum()
 fn = ((ai_detected_defect == False) & (inspector_confirmed_defect == True)).sum() # Escape
 fp = ((ai_detected_defect == True) & (inspector_confirmed_defect == False)).sum() # Overkill
 tn = ((ai_detected_defect == False) & (inspector_confirmed_defect == False)).sum()
 
 detection_rate = tp / (tp + fn) if (tp + fn) > 0 else 1.0
 overkill_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
 escape_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
 
 # Override Analysis
 override_count = sum(1 for i in verified if i.inspector_override)
 override_rate = override_count / len(verified) if verified else 0
 
 return {
 'n_inspections': len(verified),
 'detection_rate': detection_rate,
 'escape_rate': escape_rate,
 'overkill_rate': overkill_rate,
 'override_rate': override_rate,
 'meets_detection_threshold': detection_rate >= self.min_detection_rate,
 'meets_overkill_threshold': overkill_rate <= self.max_overkill_rate,
 'confusion_matrix': {'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn},
 }
 
 def generate_quality_report(self, period: str) -> str:
 """ Report Generation"""
 
 perf = self.calculate_performance()
 
 report = f"""
# {self.system_name} AI Performance Report

Period: {period} 
 : {perf.get('n_inspections', 0):,} 

---

## 1. Performance Metric

| Metric | | | |
|------|-----|------|------|
| | {perf.get('detection_rate', 0):.2%} | >={self.min_detection_rate:.0%} | {'[PASS]' if perf.get('meets_detection_threshold', False) else '[FAIL]'} |
| Escape Rate | {perf.get('escape_rate', 0):.2%} | Minimum | - |
| Overkill Rate | {perf.get('overkill_rate', 0):.2%} | <={self.max_overkill_rate:.0%} | {'[PASS]' if perf.get('meets_overkill_threshold', False) else '[WARN]'} |
| Override | {perf.get('override_rate', 0):.2%} | | - |

---

## 2. | | AI | AI |
|--|-------------|---------------|
|   | {perf.get('confusion_matrix', {}).get('tp', 0)} | {perf.get('confusion_matrix', {}).get('fn', 0)} (Escape) |
|   | {perf.get('confusion_matrix', {}).get('fp', 0)} (Overkill) | {perf.get('confusion_matrix', {}).get('tn', 0)} |

---

## 3. Impact Analysis

"""
 
 escape_count = perf.get('confusion_matrix', {}).get('fn', 0)
 if escape_count > 0:
 report += f"""
[WARN] Escape: {escape_count} 

AI .
 Analysis .

"""
 else:
 report += "[PASS] Escape 0 : AI 1 .\n\n"
 
 report += f"""
---

## 4. """
 
 if not perf.get('meets_detection_threshold', True):
 report += "- [FAIL] : Model Learning \n"
 
 if perf.get('overkill_rate', 0) > self.max_overkill_rate:
 report += "- [WARN] Overkill : ( , )\n"
 
 if perf.get('override_rate', 0) > 0.15:
 report += f"- [WARN] Override {perf['override_rate']:.1%}: AI- days Analysis \n"
 
 report += """

---

 : Management 
 : QA 
Approval: 

* Report .*
* 15 5years .*
"""
 
 return report

### 성능 및 교훈

### 제7장 요약